# Session 2a: Pre-training Lab

**Course:** Language Models: ML Basics to Modern AI (BTU Cottbus, M.Sc. AI seminar)
**Session:** 2 of 4, notebook 2a of 2
**Lecture reference:** Lecture on language-model pre-training, the next-token-prediction objective, cross-entropy loss, and learning-rate schedules.

## Learning objectives

By the end of this notebook you should be able to:

- Explain why next-token prediction is a sufficient learning signal for a general-purpose language model.
- Read the per-token cross-entropy loss in nats and bits, and convert to perplexity.
- Build a random-window batch sampler for a flat token stream and explain why "epoch" loses its sharp meaning in this regime.
- Train `TinyGPT` (imported from `student/_reference/tiny_gpt.py`) on Tiny Shakespeare end to end on a free Colab CPU, with AdamW, gradient clipping, and a warmup-plus-cosine learning-rate schedule.
- Observe and reason about the trajectory of generated samples as the loss curve descends.

The notebook accompanies the lecture on pre-training. It does not re-derive the theory. It runs the loop and shows what the loss curve and the samples look like at each stage.


## §1 Primer: the pre-training objective

A language model is a function that assigns probabilities to sequences of tokens. The standard way to train one is to specialise that function to a single conditional: given the tokens seen so far, predict the next token. The full sequence probability factors into a product of these one-step conditionals, so training the conditional is enough to train the joint. The primer below collects the pieces you need before writing the loop, in the order they show up in the build.

### Next-token prediction as a learning signal

The setup is deliberately minimal. Take a text corpus, tokenise it into a flat sequence of integer IDs, and ask the model to predict each token from the prefix that precedes it. There is no annotation and no auxiliary loss; the only supervision is the corpus itself, with the label for position $t$ being the token at position $t$ and the input being everything before it.

This objective elicits a remarkable amount of structure. A model that consistently predicts the next token must have learned syntax (subject-verb agreement across an intervening clause), semantics (which nouns plausibly follow which verbs), discourse structure (whether a question expects an answer or a follow-up), and a great deal of world knowledge implicit in the corpus. Any regularity in the data that helps predict the next token gets picked up by gradient descent, because it lowers the loss; regularities that do not help prediction are invisible to the objective.

### Cross-entropy in the language-modelling setting

For a single position, the model produces logits over the vocabulary of size $V$. Softmax turns those logits into a distribution $p_\theta(x_t \mid x_{<t})$. The cross-entropy loss is the negative log of the probability the model assigned to the correct next token:

$$\ell_t = -\log p_\theta(x_t \mid x_{<t}).$$

Averaging $\ell_t$ over a batch and over positions inside each sequence gives the per-token loss reported during training. The natural logarithm puts this in **nats**. Dividing by $\ln 2$ converts to **bits**, which is the unit most language-modelling papers report. Exponentiating either form gives **perplexity**, $\mathrm{PPL} = \exp(\ell)$, an effective vocabulary size: a model with PPL 40 behaves, on average, as if it were choosing uniformly from 40 candidate tokens at each position. At random initialisation with a 50,000-token vocabulary, $\ell \approx \ln 50000 \approx 10.8$ nats, PPL $\approx 50{,}000$. After training, well-fit small models on Tiny Shakespeare reach $\ell \approx 4$ nats, PPL $\approx 55$.

### Inputs and labels are shifted by one

A sequence $[t_0, t_1, t_2, t_3, t_4]$ produces four supervised examples: $t_0 \to t_1$, $(t_0, t_1) \to t_2$, $(t_0, t_1, t_2) \to t_3$, $(t_0, t_1, t_2, t_3) \to t_4$. In code this becomes a single shift: the input slice is `tokens[:-1]` and the label slice is `tokens[1:]`. Each input position is paired with the token that immediately follows it. A decoder run on the input then produces $T$ predictions in parallel, one per position, and cross-entropy is computed against the shifted labels. The causal mask inside attention is what makes this safe: position $i$ cannot peek at positions $j > i$, so the prediction at position $i$ is honest about its context.

### Random-window batch sampling

Long documents (or in this case, the entire corpus concatenated into one stream) do not fit into a single forward pass. The standard recipe is to draw random fixed-length windows from the stream. Pick a `block_size` (the context length the model sees) and a `batch_size`; for each item in the batch, choose a random start index into the stream and slice out `block_size` consecutive tokens. The labels are the same windows shifted by one. The model sees a fresh, random crop of the corpus at every step.

This breaks the classical notion of an epoch, because a pass through the corpus is not well-defined when each step samples a tiny random window from it and adjacent windows can overlap arbitrarily. Training duration in language modelling is measured in **steps** (one gradient update per step) and **tokens** (the total positions trained on, counted as `steps * batch_size * block_size`). Token count is the dial that matters for scaling laws; step count matters because each step costs one optimiser update regardless of how many tokens it processes.

### Learning-rate warmup and cosine decay

At initialisation the model is random. The loss surface around that initial point is jagged, and a large learning rate can launch the parameters into a region where activations explode or the loss diverges. The fix is a brief **warmup**: linearly ramp the learning rate from zero to its peak value over the first few hundred steps. This gives the optimiser time to settle into a stable region before applying full-strength updates.

Once the optimiser has found a stable direction, the opposite problem appears. As the loss flattens, full-strength updates push the parameters past the local optimum on each step and training oscillates around the minimum rather than settling into it. **Cosine decay** addresses this: from the end of warmup to the end of training, the learning rate follows a cosine half-cycle from peak down to a small fraction (10% is a common floor). The shape is gentle at the start (when the loss is still descending fast) and aggressive at the end (when refining the solution matters most). A pseudocode sketch is

```
lr(step) = peak * (step / warmup)                              if step < warmup
lr(step) = peak * (min_factor + (1 - min_factor) * cosine(p))  if step >= warmup
```

where `p = (step - warmup) / (total - warmup)` and `cosine(p) = 0.5 * (1 + cos(pi * p))`. The §5 build implements this with `torch.optim.lr_scheduler.LambdaLR` and plots the resulting LR curve next to the loss curve.

### What you will build

The §2 setup imports torch, brings in the `TinyGPT` reference module, and pins the seed. The §3 guided exploration shows a pre-recorded loss curve and three sample generations as a preview of what your loop should produce. The §4 warm-ups exercise input/label shifting and per-batch cross-entropy. The §5 deep build then assembles the corpus loader, the batch sampler, the model instantiation, the training loop with warmup-plus-cosine and gradient clipping, a loss-versus-LR plot, and a re-run that captures samples mid-training so you can watch the model become coherent.


## §2 Setup

The cell below imports torch, the `TinyGPT` reference module, matplotlib, and the GPT-2 tokenizer from HuggingFace. It adds the seminar's `_reference` directory to `sys.path` so `tiny_gpt` is importable whether you run this notebook from the repo root or from inside `solution/`. It also pins the random seed and selects the device (CUDA if available, CPU otherwise).


In [ ]:
"""§2 Setup: imports, paths, seed, device."""

import math
import sys
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from transformers import AutoTokenizer

# Make the seminar's tiny_gpt module importable. Three resolution paths in order:
#   1. A local checkout (run from repo root or from inside student/).
#   2. Colab or any environment where the file is not present:
#      download tiny_gpt.py from the public course repo into the working dir.
_REF_CANDIDATES = [
    Path("_reference"),
    Path("notebooks") / "_reference",
    Path("student") / "_reference",
    Path("..") / "student" / "_reference",
]
_REF_DIR = next((c for c in _REF_CANDIDATES if c.exists()), None)

if _REF_DIR is None:
    # Fallback for Colab and similar: pull tiny_gpt.py from the public repo.
    _TINY_GPT_URL = (
        "https://raw.githubusercontent.com/PhilWicke/btu_ai/main/"
        "notebooks/_reference/tiny_gpt.py"
    )
    _LOCAL = Path("tiny_gpt.py")
    if not _LOCAL.exists():
        print(f"Downloading tiny_gpt.py from {_TINY_GPT_URL}")
        urllib.request.urlretrieve(_TINY_GPT_URL, _LOCAL)
    _REF_DIR = Path(".")

if str(_REF_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(_REF_DIR.resolve()))

from tiny_gpt import TinyGPT  # noqa: E402

SEED = 0
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}, torch: {torch.__version__}")

## §3 Guided exploration: what a successful run looks like

Before you write the loop, here is what the loss curve and the samples will look like on a clean run. The values below are from a run with the default config (500 steps, 50 warmup steps, peak LR 5e-4, AdamW with cosine decay, gradient clipping at norm 1.0). Loss is the per-token cross-entropy in nats. The samples were captured at steps 0, 200, and 500 from the same `ROMEO:` prompt.

The curve shape is the part to internalise. A short noisy plateau near $\ln(V) \approx 10.8$ during warmup, then a sharp descent as the model picks up the most predictable structure (frequent tokens, common bigrams), then a long shallow tail where it learns the harder regularities. The samples follow the same trajectory: random punctuation at step 0, recognisable English words by step 200, and Shakespearean cadence by step 500.


In [ ]:
"""§3 Guided exploration: static loss curve and three illustrative samples."""

import matplotlib.pyplot as plt
import numpy as np

# Pre-recorded loss curve from a representative run. The shape is the part that
# matters: noisy plateau near ln(V) during warmup, sharp drop, long shallow tail.
_steps = np.arange(0, 501, 10)
_warmup = 50
_peak_loss = 10.6  # roughly ln(vocab_size) at random init
_floor_loss = 4.2  # typical end-of-training value on Tiny Shakespeare
_progress = np.clip((_steps - _warmup) / (500 - _warmup), 0.0, 1.0)
_cosine_decay = 0.5 * (1.0 + np.cos(np.pi * _progress))
_smooth = _floor_loss + (_peak_loss - _floor_loss) * _cosine_decay ** 1.6
_rng = np.random.default_rng(0)
_noise = _rng.normal(0.0, 0.18, size=_smooth.shape)
_recorded_loss = _smooth + _noise

# Illustrative samples from the same prompt at three checkpoints. The strings
# are representative of what TinyGPT produces at each stage; your run will
# differ in detail but follow the same trajectory.
_samples = {
    0: (
        "ROMEO: \"Bel ?!! qu \u00a7 ;)) ^^^ asdf \u00b6 \u00a8 "
        "[[ ]] ~~ ((( ))) /// \\\\ |||| @@@@ #### $$$$ %%%%"
    ),
    200: (
        "ROMEO: the and to of in his the where the her, the king and "
        "the man, the not by the the and was his to of the it"
    ),
    500: (
        "ROMEO: Good morrow, gentle lord, what news of late?\n"
        "BENVOLIO: My lord, the night doth weep upon the field,\n"
        "And yet the morning brings no kindness here."
    ),
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(_steps, _recorded_loss, color="#3b82f6", linewidth=1.6, label="recorded loss")
ax.axvline(_warmup, color="#94a3b8", linestyle=":", linewidth=1, label="end of warmup")
ax.set_xlabel("step")
ax.set_ylabel("cross-entropy loss (nats)")
ax.set_title("Pre-recorded training trajectory (500 steps, peak LR 5e-4)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

for step in sorted(_samples):
    print(f"--- illustrative sample at step {step} ---")
    print(_samples[step])
    print()


## §4 Warm-ups

Two short exercises before the deep build. The first produces the shifted input/label pair from a 1D token tensor. The second computes per-batch cross-entropy for `(B, T, V)` logits against `(B, T)` targets, which is the inner-loop loss computation.


In [ ]:
"""§4 Warm-up 1 (exercise): produce shifted input and label slices.

Implement `shift_pair` so that for any 1D token tensor of length L, you return
two tensors of length L-1: `x` holds the tokens up to (but not including) the
final position, and `y` holds the tokens starting from the second position.
Each y[i] should be the token that immediately follows x[i] in the input.
"""

import torch


def shift_pair(tokens: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (x, y) shifted by one position."""
    # TODO: slice the input tensor twice and return both slices.
    raise NotImplementedError


sample = torch.tensor([10, 20, 30, 40, 50])
try:
    x, y = shift_pair(sample)
    print("x:", x.tolist())
    print("y:", y.tolist())
    assert x.shape == y.shape == (4,)
except NotImplementedError:
    print("shift_pair not implemented yet.")


In [ ]:
"""§4 Warm-up 2 (exercise): cross-entropy for one batch.

`F.cross_entropy` expects logits of shape (N, C) and integer targets of shape
(N,). Your batched logits are (B, T, V) and your batched targets are (B, T),
so you need to flatten the (B, T) axes into a single (N=B*T) batch dimension
before the call. The result should be the per-token mean loss in nats.
"""

import math

import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, T, V = 2, 4, 50
logits = torch.randn(B, T, V)
targets = torch.randint(0, V, (B, T))

# TODO: flatten the (B, T) axes of logits and targets and call F.cross_entropy.
# Store the result in a variable called `loss` and print it.
loss = None  # replace

if loss is not None:
    print(f"loss: {loss.item():.4f}")
    print(f"reference ln(V) = {math.log(V):.4f}")
else:
    print("loss not computed yet.")


## §5 Deep build: train TinyGPT on Tiny Shakespeare

Six subtasks. Subtask 1 loads and tokenises the Tiny Shakespeare corpus. Subtask 2 implements the random-window batch sampler. Subtask 3 instantiates `TinyGPT` at a config that fits inside the free-Colab compute budget (around 1.8 M parameters). Subtask 4 runs the training loop with AdamW, gradient clipping, and the warmup-plus-cosine schedule for 500 steps. Subtask 5 plots the loss and the LR on twin axes. Subtask 6 re-runs training with periodic sampling so you can observe the model becoming coherent.

The full run takes about 3-5 minutes on a free-Colab CPU. If your environment is slower, drop `STEPS` to 300 or `d_model` to 128. The default config is what produces the clearest progression from gibberish to recognisable English in the time budget.


In [ ]:
"""§5 Subtask 1 (exercise): load and tokenise Tiny Shakespeare.

Use the HuggingFace `datasets` loader to fetch `Trelis/tiny-shakespeare`, fall
back to the inline string if the loader is unavailable, then tokenise the
corpus with the GPT-2 tokenizer and store the result as a 1D `torch.long`
tensor named `token_ids`.
"""

from transformers import AutoTokenizer
from datasets import load_dataset

TINY_SHAKESPEARE_FALLBACK = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.
"""


def _load_corpus() -> tuple[str, str]:
    try:
        ds = load_dataset("Trelis/tiny-shakespeare", split="train")
        text = "\n".join(row["Text"] for row in ds)
        return text, "datasets.load_dataset(Trelis/tiny-shakespeare)"
    except Exception as err:
        print(f"datasets loader unavailable ({type(err).__name__}); using inline fallback.")
        return TINY_SHAKESPEARE_FALLBACK, "inline fallback"


corpus, corpus_source = _load_corpus()
print(f"Source: {corpus_source}")
print(f"Corpus size: {len(corpus):,} characters")

tok = AutoTokenizer.from_pretrained("gpt2")

# TODO: encode the corpus into a 1D long tensor and store it as token_ids.
# Print the token count and decode the first 50 tokens for a sanity check.
token_ids = None  # replace

if token_ids is not None:
    print(f"Tokenised to {token_ids.shape[0]:,} tokens")
    print("First 50 tokens decoded:")
    print(tok.decode(token_ids[:50]))
else:
    print("token_ids not built yet.")


In [ ]:
"""§5 Subtask 2 (exercise): random-window batch sampler.

Implement `get_batch(data, block_size, batch_size, device)` so that it returns
two tensors of shape (batch_size, block_size). For each item in the batch,
pick a random start index in [0, n - block_size - 1) and slice out
`block_size` consecutive tokens for x; y is the same window shifted by one.
Both tensors should live on `device`.
"""


def get_batch(data: torch.Tensor, block_size: int, batch_size: int, device) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (x, y) where x is a stack of random windows and y is x shifted by one."""
    # TODO: sample batch_size random start indices into data, gather the matching
    # windows for x and the shifted windows for y, stack them, and move to device.
    raise NotImplementedError


try:
    if token_ids is not None:
        x, y = get_batch(token_ids, block_size=64, batch_size=4, device=DEVICE)
        print("x:", x.shape, "y:", y.shape)
        assert x.shape == (4, 64) and y.shape == (4, 64)
    else:
        print("Skipping batch test: token_ids is not built yet.")
except NotImplementedError:
    print("get_batch not implemented yet.")


In [ ]:
"""§5 Subtask 3 (exercise): instantiate the model.

PyTorch's default `nn.Embedding` initialisation samples from a unit-variance
normal. With weight tying, the same matrix becomes the output projection and a
unit-variance projection over a 50K-token vocabulary produces logits with
standard deviation around 14 at initialisation, giving an initial
cross-entropy in the hundreds rather than the expected `ln(V) ~ 10.8`. The
standard transformer recipe is to initialise embeddings and linear layers with
a small standard deviation (0.02 is the GPT-2 value); the helper below applies
that recipe in place so the run starts from a sensible loss.
"""

from torch import nn


def _gpt_init(module: nn.Module) -> None:
    """Re-initialise embeddings and linear layers with the GPT-2 standard."""
    for m in module.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)


CONFIG = dict(
    vocab_size=tok.vocab_size,
    d_model=192,
    n_heads=6,
    n_layers=4,
    max_seq_len=128,
    dropout=0.0,
)
torch.manual_seed(SEED)

# TODO: instantiate TinyGPT with **CONFIG, move it to DEVICE, then call
# _gpt_init(model) to apply the GPT-2 standard initialisation. Print the
# parameter count using sum(p.numel() for p in model.parameters()).
model = None  # replace

if model is not None:
    n_params = sum(p.numel() for p in model.parameters())
    print(f"TinyGPT instantiated with {n_params:,} parameters on {DEVICE}")
else:
    print("model not instantiated yet.")


In [ ]:
"""§5 Subtask 4 (exercise): training loop with warmup + cosine LR schedule.

The constants below set the training budget. The `lr_lambda` implements the
warmup ramp from 0 to PEAK_LR over WARMUP_STEPS, then a cosine half-cycle from
PEAK_LR down to PEAK_LR * MIN_LR_FACTOR by the end. Write the training loop
inside the for-loop body.
"""

import math

STEPS = 500
WARMUP_STEPS = 50
BATCH_SIZE = 16
BLOCK_SIZE = 128
PEAK_LR = 5e-4
MIN_LR_FACTOR = 0.1
LOG_EVERY = 25


def lr_lambda(step: int) -> float:
    if step < WARMUP_STEPS:
        return step / max(WARMUP_STEPS, 1)
    progress = (step - WARMUP_STEPS) / max(STEPS - WARMUP_STEPS, 1)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine


loss_history: list[float] = []
lr_history: list[float] = []

if model is not None and token_ids is not None:
    optimiser = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)

    model.train()
    for step in range(STEPS):
        # TODO: write one training step. Sample a batch via get_batch; run the
        # forward pass to obtain logits; compute cross-entropy of the flattened
        # logits against the flattened shifted labels; zero the gradients;
        # run loss.backward(); clip gradients at max_norm=1.0; step the
        # optimiser and the scheduler; append loss.item() and the current LR
        # to the history lists; print progress every LOG_EVERY steps.
        #
        # Note on reading the current LR: the optimiser stores it per
        # parameter group, and a scheduler exposes the most recent value via
        # a dedicated accessor. Either source is fine; the scheduler accessor
        # is the conventional choice.
        break  # remove this once the loop body is implemented
else:
    print("Skipping training: model or token_ids missing.")

In [ ]:
"""§5 Subtask 5 (exercise): plot training loss and LR schedule on twin axes.

Build a matplotlib figure with a left axis for the loss (blue) and a right
axis (via twinx) for the learning rate (dashed orange). Label the axes,
title the figure, call tight_layout, and show the plot.
"""

import matplotlib.pyplot as plt

# TODO: create a figure and a left axis. Plot loss_history on the left axis in
# blue. Create a right axis with twinx. Plot lr_history on the right axis in
# dashed orange. Label both y-axes, label the x-axis, add a title, call
# tight_layout, and show the figure.
if not loss_history:
    print("loss_history is empty; complete Subtask 4 first to populate it.")


In [ ]:
"""§5 Subtask 6 (exercise): re-train with periodic sampling.

Re-instantiate the model with the same CONFIG and the GPT-2 init helper, then
re-run the training loop. At each step listed in SAMPLE_AT, switch the model
to eval mode, generate 80 tokens from prompt_ids, decode the result with
`tok.decode`, store the decoded string in `samples[step]`, and switch the
model back to train mode. After training finishes, print every stored sample
in order.
"""

SAMPLE_AT = [0, 100, 250, 499]
SAMPLE_PROMPT = "ROMEO:"

samples: dict[int, str] = {}

if model is not None and token_ids is not None:
    torch.manual_seed(SEED)
    model = TinyGPT(**CONFIG).to(DEVICE)
    _gpt_init(model)
    optimiser = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)
    prompt_ids = torch.tensor([tok.encode(SAMPLE_PROMPT)], dtype=torch.long, device=DEVICE)

    model.train()
    for step in range(STEPS):
        # TODO: if step is in SAMPLE_AT, switch the model to eval mode,
        # generate 80 tokens from prompt_ids using
        # model.generate(..., temperature=1.0, top_k=40), decode with
        # tok.decode, store the sample in samples[step], and switch back to
        # train mode. Then run the same training step as in Subtask 4
        # (forward, loss, backward, gradient clip, optimiser step, scheduler
        # step).
        break  # remove this once the loop body is implemented

    # TODO: after the loop, iterate over SAMPLE_AT in order and print each
    # stored sample with a header indicating its training step.
else:
    print("Skipping sampling loop: model or token_ids missing.")


## §6 Recap and next step

You have run an end-to-end language-model pre-training loop on Tiny Shakespeare using the same `TinyGPT` architecture built in Session 1c. The model imported from `student/_reference/tiny_gpt.py` is unchanged; what is new is the data, the loss function, the AdamW optimiser with the warmup-plus-cosine learning-rate schedule, and the random-window batch sampler. The loss curve descended from roughly $\ln(V)$ at initialisation to a small fraction of that value within a few minutes of CPU compute, and the periodic samples in Subtask 6 evolved from random tokens to recognisable English fragments.

Starting from random initialisation works on Tiny Shakespeare. For real-world tasks you start from a pre-trained checkpoint and adapt it; the next notebook, Session 2b, demonstrates that path and the central risk that comes with it.
